# Importing all packages needed

In [1]:
import numpy as np 
import pandas as pd 


import os 
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
from datetime import datetime, timedelta
import random
import kagglehub
from __future__ import annotations

import random

# Defining the datasets

Call Center Operations Synthetic Data Generator

Purpose:
Creates a coherent, timeline-aware synthetic dataset for a call center operations analytics portfolio project.

Generated outputs:
- dim_agents.csv
- fact_schedule.csv
- fact_activity_timeline.csv
- fact_calls.csv
- fact_offline_work.csv
- fact_cases.csv
- fact_qa.csv
- fact_attendance.csv
- dim_date.csv
- dim_queue.csv

Design rules:
- All events occur inside scheduled shifts.
- Calls, offline work, case work, breaks, and lunches align on the same agent-days.
- QA reviews attach to actual generated calls.
- Attendance is derived from schedule + generated behavior.

## Config

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = "data_clean"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_AGENTS = 75
START_DATE = pd.Timestamp("2025-01-01")
END_DATE = pd.Timestamp("2025-03-31")
DATES = pd.date_range(START_DATE, END_DATE, freq="D")

TEAMS = ["Intake", "Billing", "Claims", "Technical Support"]
SUPERVISORS = ["Sup Smith", "Sup Johnson", "Sup Lee", "Sup Garcia", "Sup Patel"]
MANAGERS = ["Mgr Adams", "Mgr Brooks", "Mgr Chen"]
LOCATIONS = ["Remote", "Roanoke", "Charlotte", "Phoenix"]

SHIFT_OPTIONS = [
    ("07:00", "16:00"),
    ("08:00", "17:00"),
    ("09:00", "18:00"),
    ("10:00", "19:00"),
]

QUEUES = [
    {"queue": "Member Services", "base_call_minutes": 7, "sla_seconds": 60},
    {"queue": "Billing Support", "base_call_minutes": 9, "sla_seconds": 90},
    {"queue": "Claims Support", "base_call_minutes": 11, "sla_seconds": 120},
    {"queue": "Technical Support", "base_call_minutes": 13, "sla_seconds": 180},
]

OFFLINE_CATEGORIES = {
    "Training": {"productive": 1, "approved": 1, "min": 30, "max": 120, "weight": 0.07},
    "Team Meeting": {"productive": 1, "approved": 1, "min": 15, "max": 60, "weight": 0.10},
    "Coaching": {"productive": 1, "approved": 1, "min": 15, "max": 45, "weight": 0.08},
    "Admin": {"productive": 1, "approved": 1, "min": 10, "max": 45, "weight": 0.10},
    "Email Support": {"productive": 1, "approved": 1, "min": 10, "max": 60, "weight": 0.10},
    "Waiting for Work": {"productive": 0, "approved": 1, "min": 5, "max": 50, "weight": 0.14},
    "System Issue": {"productive": 0, "approved": 1, "min": 10, "max": 75, "weight": 0.08},
    "Break": {"productive": 0, "approved": 1, "min": 10, "max": 15, "weight": 0.16},
    "Lunch": {"productive": 0, "approved": 1, "min": 30, "max": 30, "weight": 0.00},
    "Unplanned Offline": {"productive": 0, "approved": 0, "min": 5, "max": 45, "weight": 0.07},
}

CASE_TYPES = ["Eligibility", "Prior Authorization", "Appeal", "Documentation", "Follow Up"]
CASE_COMPLEXITY = ["Low", "Medium", "High"]

## Helper Functions

In [3]:
def weighted_choice(items: list[str], weights: list[float]) -> str:
    return random.choices(items, weights=weights, k=1)[0]


def minutes_between(start: pd.Timestamp, end: pd.Timestamp) -> float:
    return (end - start).total_seconds() / 60


def clip_time(event_start: pd.Timestamp, event_end: pd.Timestamp, shift_end: pd.Timestamp):
    if event_end > shift_end:
        event_end = shift_end
    if event_start >= event_end:
        return None, None
    return event_start, event_end


def random_timestamp_between(start: pd.Timestamp, end: pd.Timestamp) -> pd.Timestamp:
    total_minutes = int(minutes_between(start, end))
    if total_minutes <= 0:
        return start
    return start + timedelta(minutes=int(np.random.randint(0, total_minutes + 1)))

## Creating the Agent Table 

In [4]:
def create_agents() -> pd.DataFrame:
    agents = pd.DataFrame({
        "master_agent_id": [f"A{str(i).zfill(3)}" for i in range(1, NUM_AGENTS + 1)],
        "agent_name": [f"Agent {i}" for i in range(1, NUM_AGENTS + 1)],
        "team": np.random.choice(TEAMS, NUM_AGENTS),
        "supervisor": np.random.choice(SUPERVISORS, NUM_AGENTS),
        "manager": np.random.choice(MANAGERS, NUM_AGENTS),
        "location": np.random.choice(LOCATIONS, NUM_AGENTS, p=[0.55, 0.20, 0.15, 0.10]),
        "remote_flag": None,
        "hire_date": START_DATE - pd.to_timedelta(np.random.randint(60, 1800, NUM_AGENTS), unit="D"),
        "employment_status": np.random.choice(["Active", "Active", "Active", "Leave"], NUM_AGENTS),
    })

    agents["remote_flag"] = (agents["location"] == "Remote").astype(int)
    agents["tenure_days"] = (START_DATE - agents["hire_date"]).dt.days
    agents["tenure_bucket"] = pd.cut(
        agents["tenure_days"],
        bins=[0, 90, 180, 365, 730, 9999],
        labels=["0-90 days", "91-180 days", "181-365 days", "1-2 years", "2+ years"],
        include_lowest=True,
    )

    # Stable default shift per agent
    shifts = [random.choice(SHIFT_OPTIONS) for _ in range(NUM_AGENTS)]
    agents["default_shift_start"] = [s[0] for s in shifts]
    agents["default_shift_end"] = [s[1] for s in shifts]
    print(agents.head())
    return agents

## Creating the Schedule Table

In [5]:
def create_schedule(agents: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, agent in agents.iterrows():
        for date in DATES:
            # Most agents work weekdays. Some weekend coverage.
            if date.weekday() >= 5 and np.random.rand() > 0.18:
                continue

            # Occasional PTO / no scheduled shift.
            if np.random.rand() < 0.025:
                continue

            shift_start = pd.Timestamp(f"{date.date()} {agent['default_shift_start']}")
            shift_end = pd.Timestamp(f"{date.date()} {agent['default_shift_end']}")
            lunch_start = shift_start + timedelta(hours=4)
            lunch_end = lunch_start + timedelta(minutes=30)

            rows.append({
                "schedule_id": f"S{len(rows) + 1:07d}",
                "master_agent_id": agent["master_agent_id"],
                "schedule_date": date.date(),
                "scheduled_start": shift_start,
                "scheduled_end": shift_end,
                "scheduled_minutes": int(minutes_between(shift_start, shift_end)),
                "planned_lunch_start": lunch_start,
                "planned_lunch_end": lunch_end,
                "planned_lunch_minutes": 30,
            })
    print(pd.DataFrame(rows).head())
    return pd.DataFrame(rows)

## Timeline Dataset

In [6]:
def create_activity_timeline(schedule: pd.DataFrame, agents: pd.DataFrame) -> pd.DataFrame:
    events = []
    activity_id = 1

    agent_team_lookup = agents.set_index("master_agent_id")["team"].to_dict()

    for _, shift in schedule.iterrows():
        agent_id = shift["master_agent_id"]
        team = agent_team_lookup[agent_id]
        current_time = pd.Timestamp(shift["scheduled_start"])
        shift_end = pd.Timestamp(shift["scheduled_end"])
        lunch_start = pd.Timestamp(shift["planned_lunch_start"])
        lunch_end = pd.Timestamp(shift["planned_lunch_end"])
        date = pd.Timestamp(shift["schedule_date"])

        # Attendance behavior flags for timeline realism.
        late_minutes = int(np.random.choice([0, 0, 0, 0, 5, 10, 15, 30], p=[0.55, 0.15, 0.10, 0.05, 0.06, 0.04, 0.03, 0.02]))
        early_leave_minutes = int(np.random.choice([0, 0, 0, 5, 10, 15, 30], p=[0.70, 0.10, 0.08, 0.04, 0.035, 0.025, 0.02]))
        absent_flag = 1 if np.random.rand() < 0.035 else 0

        if absent_flag:
            # No activities if fully absent. Attendance table will capture it.
            continue

        current_time += timedelta(minutes=late_minutes)
        actual_shift_end = shift_end - timedelta(minutes=early_leave_minutes)

        # Add a start-of-day admin block sometimes.
        if np.random.rand() < 0.45:
            dur = int(np.random.randint(5, 21))
            event_start, event_end = clip_time(current_time, current_time + timedelta(minutes=dur), actual_shift_end)
            if event_start:
                events.append({
                    "activity_id": f"ACT{activity_id:08d}",
                    "master_agent_id": agent_id,
                    "activity_date": date.date(),
                    "activity_start": event_start,
                    "activity_end": event_end,
                    "activity_type": "Offline Work",
                    "activity_category": "Admin",
                    "queue": None,
                    "case_type": None,
                    "productive_flag": 1,
                    "approved_flag": 1,
                    "source_system": "SharePoint Tracker",
                })
                activity_id += 1
                current_time = event_end

        # Generate events sequentially through the day.
        while current_time < actual_shift_end:
            # Force lunch at planned time if we reach it.
            if current_time < lunch_start < actual_shift_end:
                # Fill gap before lunch with regular activities.
                next_boundary = lunch_start
            else:
                next_boundary = actual_shift_end

            # Insert lunch if current time is at/after lunch start but before lunch end.
            if lunch_start <= current_time < lunch_end:
                event_start, event_end = clip_time(lunch_start, lunch_end, actual_shift_end)
                if event_start:
                    events.append({
                        "activity_id": f"ACT{activity_id:08d}",
                        "master_agent_id": agent_id,
                        "activity_date": date.date(),
                        "activity_start": event_start,
                        "activity_end": event_end,
                        "activity_type": "Offline Work",
                        "activity_category": "Lunch",
                        "queue": None,
                        "case_type": None,
                        "productive_flag": 0,
                        "approved_flag": 1,
                        "source_system": "Queue Manager",
                    })
                    activity_id += 1
                    current_time = event_end
                    continue

            remaining_minutes = minutes_between(current_time, next_boundary)
            if remaining_minutes <= 0:
                if current_time < lunch_end:
                    current_time = lunch_end
                else:
                    break

            # Activity mix. Technical and claims teams get more case/email work.
            if team in ["Claims", "Technical Support"]:
                activity_choices = ["Phone Call", "Case Work", "Offline Work", "Idle"]
                activity_weights = [0.55, 0.20, 0.20, 0.05]
            else:
                activity_choices = ["Phone Call", "Case Work", "Offline Work", "Idle"]
                activity_weights = [0.70, 0.08, 0.17, 0.05]

            activity_type = weighted_choice(activity_choices, activity_weights)

            if activity_type == "Phone Call":
                queue_info = random.choice(QUEUES)
                talk_minutes = max(2, np.random.normal(queue_info["base_call_minutes"], 3))
                hold_minutes = max(0, np.random.exponential(1.2))
                acw_minutes = max(1, np.random.normal(3, 1.2))
                dur = int(round(talk_minutes + hold_minutes + acw_minutes))
                dur = max(3, min(dur, 35))

                event_start, event_end = clip_time(current_time, current_time + timedelta(minutes=dur), next_boundary)
                if not event_start:
                    current_time += timedelta(minutes=1)
                    continue

                events.append({
                    "activity_id": f"ACT{activity_id:08d}",
                    "master_agent_id": agent_id,
                    "activity_date": date.date(),
                    "activity_start": event_start,
                    "activity_end": event_end,
                    "activity_type": "Phone Call",
                    "activity_category": "Inbound Call",
                    "queue": queue_info["queue"],
                    "case_type": None,
                    "productive_flag": 1,
                    "approved_flag": 1,
                    "source_system": "Queue Manager",
                })
                activity_id += 1
                current_time = event_end

            elif activity_type == "Case Work":
                dur = int(np.random.randint(8, 46))
                event_start, event_end = clip_time(current_time, current_time + timedelta(minutes=dur), next_boundary)
                if not event_start:
                    current_time += timedelta(minutes=1)
                    continue

                events.append({
                    "activity_id": f"ACT{activity_id:08d}",
                    "master_agent_id": agent_id,
                    "activity_date": date.date(),
                    "activity_start": event_start,
                    "activity_end": event_end,
                    "activity_type": "Case Work",
                    "activity_category": "Case Processing",
                    "queue": None,
                    "case_type": random.choice(CASE_TYPES),
                    "productive_flag": 1,
                    "approved_flag": 1,
                    "source_system": "Salesforce",
                })
                activity_id += 1
                current_time = event_end

            elif activity_type == "Offline Work":
                cats = list(OFFLINE_CATEGORIES.keys())
                cats = [c for c in cats if c != "Lunch"]
                weights = [OFFLINE_CATEGORIES[c]["weight"] for c in cats]
                category = weighted_choice(cats, weights)
                info = OFFLINE_CATEGORIES[category]
                dur = int(np.random.randint(info["min"], info["max"] + 1))

                event_start, event_end = clip_time(current_time, current_time + timedelta(minutes=dur), next_boundary)
                if not event_start:
                    current_time += timedelta(minutes=1)
                    continue

                source_system = "Queue Manager"
                if category in ["Training", "Team Meeting", "Coaching", "Admin"]:
                    source_system = "SharePoint Tracker"
                if category == "System Issue":
                    source_system = "IT Incident Log"

                events.append({
                    "activity_id": f"ACT{activity_id:08d}",
                    "master_agent_id": agent_id,
                    "activity_date": date.date(),
                    "activity_start": event_start,
                    "activity_end": event_end,
                    "activity_type": "Offline Work",
                    "activity_category": category,
                    "queue": None,
                    "case_type": None,
                    "productive_flag": info["productive"],
                    "approved_flag": info["approved"],
                    "source_system": source_system,
                })
                activity_id += 1
                current_time = event_end

            else:
                # Idle/waiting gap. Keep short.
                dur = int(np.random.randint(2, 16))
                event_start, event_end = clip_time(current_time, current_time + timedelta(minutes=dur), next_boundary)
                if not event_start:
                    current_time += timedelta(minutes=1)
                    continue

                events.append({
                    "activity_id": f"ACT{activity_id:08d}",
                    "master_agent_id": agent_id,
                    "activity_date": date.date(),
                    "activity_start": event_start,
                    "activity_end": event_end,
                    "activity_type": "Offline Work",
                    "activity_category": "Waiting for Work",
                    "queue": None,
                    "case_type": None,
                    "productive_flag": 0,
                    "approved_flag": 1,
                    "source_system": "Queue Manager",
                })
                activity_id += 1
                current_time = event_end

            # Small natural gap between activities sometimes.
            if np.random.rand() < 0.15:
                current_time += timedelta(minutes=int(np.random.randint(1, 4)))

    timeline = pd.DataFrame(events)
    timeline["duration_minutes"] = (
        pd.to_datetime(timeline["activity_end"]) - pd.to_datetime(timeline["activity_start"])
    ).dt.total_seconds() / 60
    print(timeline.head())
    return timeline


# =========================================================
# 4. DERIVE FACT CALLS
# =========================================================

def create_calls(timeline: pd.DataFrame) -> pd.DataFrame:
    calls = timeline[timeline["activity_type"] == "Phone Call"].copy().reset_index(drop=True)
    calls["call_id"] = [f"C{i + 1:08d}" for i in range(len(calls))]

    # Split duration into talk/hold/acw in a realistic-ish way.
    duration = calls["duration_minutes"].astype(float)
    calls["hold_seconds"] = np.round(np.maximum(0, np.random.exponential(45, len(calls)))).astype(int)
    calls["after_call_work_seconds"] = np.round(np.maximum(30, np.random.normal(180, 60, len(calls)))).astype(int)
    calls["talk_seconds"] = np.round(duration * 60 - calls["hold_seconds"] - calls["after_call_work_seconds"]).astype(int)
    calls.loc[calls["talk_seconds"] < 60, "talk_seconds"] = 60

    calls["answer_speed_seconds"] = np.round(np.random.exponential(55, len(calls))).astype(int)
    calls["abandoned_flag"] = np.where(calls["answer_speed_seconds"] > 300, 1, 0)

    queue_sla = {q["queue"]: q["sla_seconds"] for q in QUEUES}
    calls["sla_seconds"] = calls["queue"].map(queue_sla)
    calls["sla_met_flag"] = np.where(calls["answer_speed_seconds"] <= calls["sla_seconds"], 1, 0)
    calls["customer_id"] = [f"CU{np.random.randint(1, 20000):07d}" for _ in range(len(calls))]

    calls = calls[[
        "call_id",
        "activity_id",
        "master_agent_id",
        "activity_date",
        "activity_start",
        "activity_end",
        "queue",
        "talk_seconds",
        "hold_seconds",
        "after_call_work_seconds",
        "answer_speed_seconds",
        "sla_seconds",
        "sla_met_flag",
        "abandoned_flag",
        "customer_id",
    ]].rename(columns={
        "activity_date": "call_date",
        "activity_start": "call_start",
        "activity_end": "call_end",
    })

    calls["aht_seconds"] = calls["talk_seconds"] + calls["hold_seconds"] + calls["after_call_work_seconds"]
    print(calls.head())
    return calls


# =========================================================
# 5. DERIVE FACT OFFLINE WORK
# =========================================================

def create_offline_work(timeline: pd.DataFrame) -> pd.DataFrame:
    offline = timeline[timeline["activity_type"] == "Offline Work"].copy().reset_index(drop=True)
    offline["offline_id"] = [f"O{i + 1:08d}" for i in range(len(offline))]

    offline = offline[[
        "offline_id",
        "activity_id",
        "master_agent_id",
        "activity_date",
        "activity_start",
        "activity_end",
        "activity_category",
        "productive_flag",
        "approved_flag",
        "duration_minutes",
        "source_system",
    ]].rename(columns={
        "activity_date": "offline_date",
        "activity_start": "offline_start",
        "activity_end": "offline_end",
        "activity_category": "offline_category",
    })
    print(offline.head())
    return offline


# =========================================================
# 6. DERIVE FACT CASES
# =========================================================

def create_cases(timeline: pd.DataFrame) -> pd.DataFrame:
    cases = timeline[timeline["activity_type"] == "Case Work"].copy().reset_index(drop=True)
    cases["case_id"] = [f"CS{i + 1:08d}" for i in range(len(cases))]
    cases["case_complexity"] = np.random.choice(CASE_COMPLEXITY, len(cases), p=[0.55, 0.32, 0.13])

    # Case SLA logic: high complexity has slightly lower SLA met odds.
    sla_probs = cases["case_complexity"].map({"Low": 0.94, "Medium": 0.87, "High": 0.76})
    cases["case_sla_met_flag"] = [1 if np.random.rand() < p else 0 for p in sla_probs]
    cases["case_turnaround_hours"] = np.round(
        np.where(
            cases["case_complexity"] == "High",
            np.random.normal(36, 12, len(cases)),
            np.where(cases["case_complexity"] == "Medium", np.random.normal(20, 8, len(cases)), np.random.normal(8, 4, len(cases)))
        ), 1
    )
    cases["case_turnaround_hours"] = cases["case_turnaround_hours"].clip(lower=0.5)

    cases = cases[[
        "case_id",
        "activity_id",
        "master_agent_id",
        "activity_date",
        "activity_start",
        "activity_end",
        "case_type",
        "case_complexity",
        "case_turnaround_hours",
        "case_sla_met_flag",
    ]].rename(columns={
        "activity_date": "case_date",
        "activity_start": "case_start",
        "activity_end": "case_end",
    })
    print(cases.head())
    return cases


# =========================================================
# 7. FACT QA
# =========================================================

def create_qa(calls: pd.DataFrame, agents: pd.DataFrame) -> pd.DataFrame:
    # QA reviews approx 8% of calls, max enough for portfolio.
    sample_size = max(1, int(len(calls) * 0.08))
    qa_calls = calls.sample(sample_size, random_state=SEED).copy().reset_index(drop=True)

    tenure_lookup = agents.set_index("master_agent_id")["tenure_days"].to_dict()
    qa_rows = []

    for i, row in qa_calls.iterrows():
        tenure_days = tenure_lookup.get(row["master_agent_id"], 365)
        tenure_boost = min(5, tenure_days / 365)
        base = np.random.normal(84 + tenure_boost, 7)

        qa_score = np.clip(base, 45, 100)
        compliance = np.clip(base + np.random.normal(0, 5), 40, 100)
        empathy = np.clip(base + np.random.normal(0, 6), 40, 100)
        documentation = np.clip(base + np.random.normal(0, 7), 40, 100)

        qa_rows.append({
            "qa_id": f"Q{i + 1:08d}",
            "call_id": row["call_id"],
            "master_agent_id": row["master_agent_id"],
            "review_date": (pd.Timestamp(row["call_date"]) + timedelta(days=int(np.random.randint(1, 10)))).date(),
            "qa_score": round(float(qa_score), 1),
            "compliance_score": round(float(compliance), 1),
            "empathy_score": round(float(empathy), 1),
            "documentation_score": round(float(documentation), 1),
            "qa_pass_flag": 1 if qa_score >= 80 and compliance >= 75 else 0,
        })
    return pd.DataFrame(qa_rows)


# =========================================================
# 8. FACT ATTENDANCE
# =========================================================

def create_attendance(schedule: pd.DataFrame, timeline: pd.DataFrame) -> pd.DataFrame:
    worked = timeline.groupby(["master_agent_id", "activity_date"], as_index=False)["duration_minutes"].sum()
    worked = worked.rename(columns={"activity_date": "schedule_date", "duration_minutes": "worked_minutes"})

    attendance = schedule[[
        "master_agent_id", "schedule_date", "scheduled_start", "scheduled_end", "scheduled_minutes"
    ]].copy()

    attendance["schedule_date"] = pd.to_datetime(attendance["schedule_date"]).dt.date
    worked["schedule_date"] = pd.to_datetime(worked["schedule_date"]).dt.date

    attendance = attendance.merge(worked, on=["master_agent_id", "schedule_date"], how="left")
    attendance["worked_minutes"] = attendance["worked_minutes"].fillna(0)
    attendance["absent_flag"] = np.where(attendance["worked_minutes"] == 0, 1, 0)

    # Estimate late/early from first/last event.
    first_last = timeline.groupby(["master_agent_id", "activity_date"], as_index=False).agg(
        first_activity=("activity_start", "min"),
        last_activity=("activity_end", "max"),
    ).rename(columns={"activity_date": "schedule_date"})

    first_last["schedule_date"] = pd.to_datetime(first_last["schedule_date"]).dt.date
    attendance = attendance.merge(first_last, on=["master_agent_id", "schedule_date"], how="left")

    attendance["late_minutes"] = (
        pd.to_datetime(attendance["first_activity"]) - pd.to_datetime(attendance["scheduled_start"])
    ).dt.total_seconds() / 60
    attendance["late_minutes"] = attendance["late_minutes"].fillna(0).clip(lower=0).round(0).astype(int)

    attendance["early_leave_minutes"] = (
        pd.to_datetime(attendance["scheduled_end"]) - pd.to_datetime(attendance["last_activity"])
    ).dt.total_seconds() / 60
    attendance["early_leave_minutes"] = attendance["early_leave_minutes"].fillna(attendance["scheduled_minutes"]).clip(lower=0).round(0).astype(int)

    attendance["worked_hours"] = np.round(attendance["worked_minutes"] / 60, 2)
    attendance["scheduled_hours"] = np.round(attendance["scheduled_minutes"] / 60, 2)
    attendance["adherence_pct"] = np.where(
        attendance["scheduled_minutes"] > 0,
        np.round(attendance["worked_minutes"] / attendance["scheduled_minutes"], 3),
        np.nan,
    )

    attendance["attendance_id"] = [f"AT{i + 1:08d}" for i in range(len(attendance))]

    attendance = attendance[[
        "attendance_id",
        "master_agent_id",
        "schedule_date",
        "scheduled_hours",
        "worked_hours",
        "scheduled_minutes",
        "worked_minutes",
        "absent_flag",
        "late_minutes",
        "early_leave_minutes",
        "adherence_pct",
    ]].rename(columns={"schedule_date": "date"})
    print(attendance.head())
    return attendance


# =========================================================
# 9. DIMENSIONS
# =========================================================

def create_date_dim() -> pd.DataFrame:
    dim_date = pd.DataFrame({"date": DATES.date})
    dim_date["date"] = pd.to_datetime(dim_date["date"])
    dim_date["year"] = dim_date["date"].dt.year
    dim_date["month"] = dim_date["date"].dt.month
    dim_date["month_name"] = dim_date["date"].dt.month_name()
    dim_date["week"] = dim_date["date"].dt.isocalendar().week.astype(int)
    dim_date["day_of_week"] = dim_date["date"].dt.day_name()
    dim_date["is_weekend"] = dim_date["date"].dt.weekday >= 5
    dim_date["date"] = dim_date["date"].dt.date
    print(dim_date.head())
    return dim_date


def create_queue_dim() -> pd.DataFrame:
    return pd.DataFrame(QUEUES)


# =========================================================
# 10. VALIDATION
# =========================================================

def validate_outputs(schedule: pd.DataFrame, timeline: pd.DataFrame, calls: pd.DataFrame, qa: pd.DataFrame):
    print("\nVALIDATION")
    print("==========")
    print(f"Scheduled agent-days: {len(schedule):,}")
    print(f"Timeline activities:  {len(timeline):,}")
    print(f"Calls:                {len(calls):,}")
    print(f"QA reviews:           {len(qa):,}")

    # Check events are inside schedule.
    check = timeline.merge(
        schedule[["master_agent_id", "schedule_date", "scheduled_start", "scheduled_end"]],
        left_on=["master_agent_id", "activity_date"],
        right_on=["master_agent_id", "schedule_date"],
        how="left",
    )
    bad = check[
        (pd.to_datetime(check["activity_start"]) < pd.to_datetime(check["scheduled_start"])) |
        (pd.to_datetime(check["activity_end"]) > pd.to_datetime(check["scheduled_end"]))
    ]
    print(f"Events outside scheduled shift: {len(bad):,}")

    print("\nActivity mix:")
    print(timeline["activity_category"].value_counts().head(15))

    print("\nQueue mix:")
    print(calls["queue"].value_counts())

    print("\nSLA rate:")
    print(round(calls["sla_met_flag"].mean(), 3))


# =========================================================
# 11. MAIN
# =========================================================

def main():
    agents = create_agents()
    schedule = create_schedule(agents)
    timeline = create_activity_timeline(schedule, agents)
    calls = create_calls(timeline)
    offline = create_offline_work(timeline)
    cases = create_cases(timeline)
    qa = create_qa(calls, agents)
    attendance = create_attendance(schedule, timeline)
    dim_date = create_date_dim()
    dim_queue = create_queue_dim()

    outputs = {
        "dim_agents.csv": agents,
        "fact_schedule.csv": schedule,
        "fact_activity_timeline.csv": timeline,
        "fact_calls.csv": calls,
        "fact_offline_work.csv": offline,
        "fact_cases.csv": cases,
        "fact_qa.csv": qa,
        "fact_attendance.csv": attendance,
        "dim_date.csv": dim_date,
        "dim_queue.csv": dim_queue,
    }

    for filename, df in outputs.items():
        path = os.path.join(OUTPUT_DIR, filename)
        df.to_csv(path, index=False)
        print(f"Saved {filename}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

    validate_outputs(schedule, timeline, calls, qa)


if __name__ == "__main__":
    main()

  master_agent_id agent_name               team   supervisor    manager  \
0            A001    Agent 1             Claims  Sup Johnson   Mgr Chen   
1            A002    Agent 2  Technical Support   Sup Garcia   Mgr Chen   
2            A003    Agent 3             Intake   Sup Garcia   Mgr Chen   
3            A004    Agent 4             Claims   Sup Garcia  Mgr Adams   
4            A005    Agent 5             Claims   Sup Garcia  Mgr Adams   

  location  remote_flag  hire_date employment_status  tenure_days  \
0  Roanoke            0 2020-03-15            Active         1753   
1  Roanoke            0 2024-09-26            Active           97   
2   Remote            1 2024-03-18            Active          289   
3   Remote            1 2021-01-14            Active         1448   
4  Roanoke            0 2023-04-20            Active          622   

  tenure_bucket default_shift_start default_shift_end  
0      2+ years               07:00             16:00  
1   91-180 days       